In [ ]:
# ============================================================
# DPO (QLoRA) for Qwen/Qwen3-8B: Learn correct relaxation choice
# ============================================================

import os, json, random, re, math, inspect, subprocess, sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# -------- GPU selection (default: single GPU) --------
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "5")

# -------- pip installs (quiet) --------
def _pip_install(pkgs: List[str]):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

def _version_tuple(v: str):
    nums = []
    for x in re.findall(r"\d+", v.split("+")[0]):
        nums.append(int(x))
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums[:3])

try:
    import torch
except Exception:
    _pip_install(["torch"])
    import torch

try:
    import transformers
except Exception:
    _pip_install(["transformers"])
    import transformers

# Qwen3 likes newer transformers; keep this conservative
import transformers as _tf
if _version_tuple(_tf.__version__) < (4, 45, 0):
    _pip_install(["-U", "transformers"])
    import importlib
    importlib.reload(_tf)
    import transformers

try:
    import datasets
except Exception:
    _pip_install(["datasets"])
    import datasets

try:
    import peft
except Exception:
    _pip_install(["peft"])
    import peft

try:
    import bitsandbytes
except Exception:
    _pip_install(["bitsandbytes"])
    import bitsandbytes

# TRL for DPO
try:
    import trl
except Exception:
    _pip_install(["trl"])
    import trl

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOTrainer

# -------- Repro --------
SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

# -------- Paths --------
TRAIN_PATH_CANDS = ["./bench_train_80.json", "/mnt/data/bench_train_80.json"]
TEST_PATH_CANDS  = ["./bench_test_20.json",  "/mnt/data/bench_test_20.json"]
DATA_PATH_CANDS  = ["./data.csv", "/mnt/data/data.csv"]

ROLEPLAYER_ADAPTER_DIR_CANDS = ["./qwen3_roleplayer_sft_lora", "/mnt/data/qwen3_roleplayer_sft_lora"]

OUT_DIR = "./qwen3_agent_dpo_relax_lora"
PAIR_TRAIN_JSONL = "./dpo_pairs_train.jsonl"
PAIR_TEST_JSONL  = "./dpo_pairs_test.jsonl"

MODEL_NAME = "Qwen/Qwen3-8B"

train_path = next((p for p in TRAIN_PATH_CANDS if os.path.exists(p)), None)
test_path  = next((p for p in TEST_PATH_CANDS  if os.path.exists(p)), None)
data_path  = next((p for p in DATA_PATH_CANDS  if os.path.exists(p)), None)
rp_dir     = next((p for p in ROLEPLAYER_ADAPTER_DIR_CANDS if os.path.exists(p)), None)

if train_path is None:
    raise FileNotFoundError(f"Could not find bench_train_80.json in: {TRAIN_PATH_CANDS}")
if test_path is None:
    raise FileNotFoundError(f"Could not find bench_test_20.json in: {TEST_PATH_CANDS}")
if data_path is None:
    raise FileNotFoundError(f"Could not find data.csv in: {DATA_PATH_CANDS}")
if rp_dir is None:
    raise FileNotFoundError(
        f"Could not find roleplayer SFT adapter directory in: {ROLEPLAYER_ADAPTER_DIR_CANDS}\n"
        f"(Expected ./qwen3_roleplayer_sft_lora from your earlier SFT run.)"
    )

with open(train_path, "r", encoding="utf-8") as f:
    bench_train = json.load(f)
with open(test_path, "r", encoding="utf-8") as f:
    bench_test = json.load(f)

# -------- Basic validation / schema probing --------
def _has_keys(x: Dict[str,Any], keys: List[str]) -> bool:
    return all(k in x for k in keys)

need_any = ["base_query_sentence", "persona", "additional_constraints", "chosen_relaxation"]
for name, data in [("train", bench_train), ("test", bench_test)]:
    if not isinstance(data, list) or not data:
        raise ValueError(f"{name} split is not a non-empty JSON list.")
    missing = [k for k in need_any if k not in data[0]]
    if missing:
        # Allow "persona" to sometimes be absent; but training would be weaker. Still, we fail loudly.
        raise ValueError(f"{name} split item[0] missing keys: {missing}. "
                         f"Found keys: {list(data[0].keys())[:30]}")

# -------- Dataset / columns (for feasibility checks only; not shown to model) --------
import pandas as pd
import numpy as np

EXCLUDE = {"Engine HP", "Engine Cylinders", "Market Category", "Popularity", "Number of Doors"}
BASE_COLS = ["Make", "Transmission Type", "Driven_Wheels", "Vehicle Style"]

df0 = pd.read_csv(data_path)
df = df0[[c for c in df0.columns if c not in EXCLUDE]].copy()
if "Engine Fuel Type" in df.columns:
    df["Engine Fuel Type"] = df["Engine Fuel Type"].fillna("Unknown")

# -------- Helper: parse base fields from base_query_sentence by vocab match --------
# (matches your earlier pipeline style; assumes base sentence contains exact tokens)
def parse_base_from_sentence(df: pd.DataFrame, base_sentence: str) -> Dict[str,str]:
    s = (base_sentence or "").lower()
    def match_vocab(col: str) -> str:
        vocab = sorted(df[col].dropna().astype(str).unique().tolist(), key=len, reverse=True)
        for v in vocab:
            if v.lower() in s:
                return v
        raise ValueError(f"Could not parse base field {col} from: {base_sentence}")
    return {
        "Make": match_vocab("Make"),
        "Vehicle Style": match_vocab("Vehicle Style"),
        "Transmission Type": match_vocab("Transmission Type"),
        "Driven_Wheels": match_vocab("Driven_Wheels"),
    }

# -------- Helper: apply constraints (for feasibility-based negative sampling) --------
def apply_constraints(df: pd.DataFrame, base: Dict[str,str], extras: List[Dict[str,Any]]) -> pd.DataFrame:
    m = np.ones(len(df), dtype=bool)
    for c in BASE_COLS:
        m &= (df[c].values == base[c])
    for p in extras:
        col, op, val = p["column"], p["op"], p["value"]
        if col not in df.columns:
            return df.iloc[0:0].copy()
        if op == "==":
            m &= (df[col].astype(str).values == str(val))
        elif op == ">=":
            m &= (df[col].values >= int(val))
        elif op == "<=":
            m &= (df[col].values <= int(val))
        else:
            return df.iloc[0:0].copy()
    return df[m].copy()

# -------- Roleplayer prompt templates (must match what you SFT'ed) --------
ALL_SLOTS = ["Year","MSRP","city mpg","highway MPG","Engine Fuel Type","Vehicle Size","Model"]

QUESTION_TEMPLATES = {
    "Year": "What is your minimum model year? Give a year number, or say you have no preference.",
    "MSRP": "What is your maximum budget (MSRP)? Give a number, or say you have no preference.",
    "city mpg": "What minimum city MPG do you want? Give a number, or say you have no preference.",
    "highway MPG": "What minimum highway MPG do you want? Give a number, or say you have no preference.",
    "Engine Fuel Type": "Do you have a preferred engine fuel type? If yes, state it; otherwise say no preference.",
    "Vehicle Size": "Do you have a preferred vehicle size (e.g., Compact, Midsize, Large)? If yes, state it; otherwise say no preference.",
    "Model": "Do you have a specific model in mind? If yes, state it; otherwise say no preference.",
}

ROLEPLAYER_SFT_SYS = (
    "You are ROLEPLAYER_USER.\n"
    "You ONLY know the provided persona and the base query.\n"
    "You must answer the assistant's question using ONLY information stated or clearly implied in the persona.\n"
    "CRITICAL:\n"
    " - If the persona includes a concrete value relevant to the asked slot, you MUST state that value explicitly.\n"
    " - Only if the persona truly contains no information about that slot, say you have no preference.\n"
    "Output MUST be valid JSON ONLY, matching the given schema.\n"
    "Do NOT include any extra keys.\n"
)

def roleplayer_user_prompt(base_query: str, persona: str, slot: str) -> str:
    q = QUESTION_TEMPLATES.get(slot, f"What is your preference for {slot}?")
    return (
        f"Base query:\n{base_query}\n\n"
        f"Persona:\n{persona}\n\n"
        f"Assistant question (slot={slot}):\n{q}\n\n"
        "Return ONLY JSON in this schema:\n"
        '{\n'
        '  "slot": "<slot>",\n'
        '  "has_preference": true/false,\n'
        '  "op": "=="|">="|"<=",\n'
        '  "value": string|int|null,\n'
        '  "importance_label": "MUST_HAVE"|"STRONG_PREFERENCE"|"NICE_TO_HAVE"|"FLEXIBLE"|"NO_PREFERENCE"\n'
        '}\n'
    )

# -------- Robust JSON extraction from model text --------
def extract_json_robust(text: str) -> Optional[dict]:
    t = (text or "").strip()
    if not t:
        return None
    # if fenced
    if "```" in t:
        parts = t.split("```")
        for i in range(len(parts)-1):
            payload = parts[i+1].strip()
            # strip leading json tag
            lines = payload.splitlines()
            if lines and lines[0].strip().lower() == "json":
                payload = "\n".join(lines[1:]).strip()
            try:
                return json.loads(payload)
            except Exception:
                pass
    # scan outermost object
    start = t.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(t)):
        if t[i] == "{":
            depth += 1
        elif t[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(t[start:i+1])
                except Exception:
                    return None
    return None

# ============================================================
# Load tokenizer (shared)
# ============================================================
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# ============================================================
# Load Roleplayer generator model (base + your SFT adapter)
# ============================================================
rp_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
rp_model = PeftModel.from_pretrained(rp_base, rp_dir)
rp_model.eval()

@torch.inference_mode()
def roleplayer_generate_json(base_query: str, persona: str, slot: str, max_new_tokens: int = 160) -> dict:
    msgs = [
        {"role":"system", "content": ROLEPLAYER_SFT_SYS},
        {"role":"user", "content": roleplayer_user_prompt(base_query, persona, slot)},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(rp_model.device)
    out = rp_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    obj = extract_json_robust(gen) or {
        "slot": slot, "has_preference": False, "op": "==", "value": None, "importance_label": "NO_PREFERENCE"
    }
    # enforce minimal schema
    obj.setdefault("slot", slot)
    obj.setdefault("has_preference", False)
    obj.setdefault("op", "==")
    obj.setdefault("value", None)
    obj.setdefault("importance_label", "NO_PREFERENCE")
    return obj

# ============================================================
# Build DPO relaxation pairs from split JSON
# ============================================================
AGENT_SYS = (
    "You are an assistant resolving an INFEASIBLE structured query against a database.\n"
    "You are given a short conversation where the user clarified preferences.\n"
    "The full set of additional constraints yields NO results.\n"
    "Choose EXACTLY ONE additional constraint to relax.\n"
    "Output ONLY JSON in this schema:\n"
    '{ "relaxed_constraint": {"column": "...", "op": "==|>=|<=", "value": "..."} }\n'
)

def _canon_relax_obj(c: Dict[str,Any]) -> Dict[str,Any]:
    return {"relaxed_constraint": {"column": c["column"], "op": c["op"], "value": c["value"]}}

def _same_constraint(a: Dict[str,Any], b: Dict[str,Any]) -> bool:
    try:
        return (a.get("column"), a.get("op"), str(a.get("value"))) == (b.get("column"), b.get("op"), str(b.get("value")))
    except Exception:
        return False

def build_prompt_text(base_query: str, persona: str, constraints: List[Dict[str,Any]]) -> str:
    # Simulate a short dialogue: one question per slot in constraints, answered by roleplayer JSON
    dialogue_lines = []
    for c in constraints:
        slot = c["column"]
        q = QUESTION_TEMPLATES.get(slot, f"What about {slot}?")
        a = roleplayer_generate_json(base_query, persona, slot)
        dialogue_lines.append(f"ASSISTANT: {q}")
        dialogue_lines.append(f"USER: {json.dumps(a, ensure_ascii=False)}")

    user_block = (
        f"Base query:\n{base_query}\n\n"
        f"Conversation:\n" + "\n".join(dialogue_lines) + "\n\n"
        "Additional constraints (ALL together yield no results):\n"
        + json.dumps(constraints, ensure_ascii=False, indent=2)
        + "\n\n"
        "Task: Choose exactly ONE constraint to relax.\n"
        "Return ONLY the JSON schema."
    )

    msgs = [{"role":"system","content":AGENT_SYS},{"role":"user","content":user_block}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

def build_dpo_pairs(split: List[Dict[str,Any]], max_negatives_per_item: int = 4) -> List[Dict[str,str]]:
    rng = random.Random(SEED)
    pairs = []
    skipped = 0

    for it in split:
        base_query = (it.get("base_query_sentence") or "").strip()
        persona = (it.get("persona") or "").strip()
        constraints = it.get("additional_constraints") or []
        oracle_relax = it.get("chosen_relaxation") or it.get("relaxed_constraint") or it.get("unique_repair_constraint")

        if not base_query or not persona or not isinstance(constraints, list) or not constraints or not isinstance(oracle_relax, dict):
            skipped += 1
            continue

        # Ensure oracle relax is one of constraints; if not, skip
        if not any(_same_constraint(c, oracle_relax) for c in constraints):
            skipped += 1
            continue

        # Feasibility-aware negative sampling (not shown to model)
        try:
            base = parse_base_from_sentence(df, base_query)
        except Exception:
            skipped += 1
            continue

        # full should be UNSAT; if SAT, skip (we're training infeasibility repair)
        full = apply_constraints(df, base, constraints)
        if len(full) > 0:
            skipped += 1
            continue

        prompt_text = build_prompt_text(base_query, persona, constraints)
        chosen = json.dumps(_canon_relax_obj(oracle_relax), ensure_ascii=False)

        # Build candidates for rejected: other constraints
        other = [c for c in constraints if not _same_constraint(c, oracle_relax)]
        if not other:
            skipped += 1
            continue

        # Rank negatives: prefer ones that stay infeasible after relaxing (hard negatives)
        neg_bins = []
        for c in other:
            keep = [x for x in constraints if not _same_constraint(x, c)]
            cand = apply_constraints(df, base, keep)
            neg_bins.append((0 if len(cand) == 0 else 1, len(cand), c))  # 0=still infeasible (hard), 1=feasible
        neg_bins.sort(key=lambda t: (t[0], t[1]))  # hard first, then smaller feasible sets

        # Sample up to max_negatives_per_item with a mix: take hard-first then a random feasible if available
        hard = [c for (flag,_,c) in neg_bins if flag == 0]
        feas = [c for (flag,_,c) in neg_bins if flag == 1]

        picked = []
        picked.extend(hard[:max_negatives_per_item])
        if len(picked) < max_negatives_per_item and feas:
            rng.shuffle(feas)
            picked.extend(feas[:(max_negatives_per_item - len(picked))])

        for rej_c in picked:
            rejected = json.dumps(_canon_relax_obj(rej_c), ensure_ascii=False)
            pairs.append({"prompt": prompt_text, "chosen": chosen, "rejected": rejected})

    print(f"Built {len(pairs)} DPO pairs (skipped={skipped}, max_negatives_per_item={max_negatives_per_item})")
    return pairs

train_pairs = build_dpo_pairs(bench_train, max_negatives_per_item=4)
test_pairs  = build_dpo_pairs(bench_test,  max_negatives_per_item=2)

# Save pairs for inspection/debugging
with open(PAIR_TRAIN_JSONL, "w", encoding="utf-8") as f:
    for r in train_pairs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(PAIR_TEST_JSONL, "w", encoding="utf-8") as f:
    for r in test_pairs:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved pair files:\n  {PAIR_TRAIN_JSONL}\n  {PAIR_TEST_JSONL}")

train_ds = Dataset.from_list(train_pairs)
test_ds  = Dataset.from_list(test_pairs)

# ============================================================
# Load policy + ref model (both 4-bit), attach LoRA to policy
# ============================================================
def guess_lora_targets(model) -> List[str]:
    preferred = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    found = set()
    for name, module in model.named_modules():
        if module.__class__.__name__ == "Linear":
            last = name.split(".")[-1]
            if last in preferred:
                found.add(last)
    return sorted(found) if found else preferred

policy = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
policy.config.use_cache = False
policy = prepare_model_for_kbit_training(policy, use_gradient_checkpointing=True)

target_modules = guess_lora_targets(policy)
print("LoRA target_modules:", target_modules)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)
policy = get_peft_model(policy, lora_cfg)
policy.print_trainable_parameters()

# Reference model: same base, no LoRA, frozen
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# ============================================================
# TrainingArguments (version-robust)
# ============================================================
sig = inspect.signature(TrainingArguments.__init__)
has_eval_strategy = "eval_strategy" in sig.parameters
has_evaluation_strategy = "evaluation_strategy" in sig.parameters
has_report_to = "report_to" in sig.parameters
has_optim = "optim" in sig.parameters

ta_kwargs = dict(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,                 # DPO tends to be data-amplified; start small
    learning_rate=1e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    save_total_limit=2,
    fp16=True,
)

if has_eval_strategy:
    ta_kwargs["eval_strategy"] = "steps"
elif has_evaluation_strategy:
    ta_kwargs["evaluation_strategy"] = "steps"

ta_kwargs["save_strategy"] = "steps"
if has_report_to:
    ta_kwargs["report_to"] = "none"
if has_optim:
    ta_kwargs["optim"] = "paged_adamw_32bit"
if "gradient_checkpointing" in sig.parameters:
    ta_kwargs["gradient_checkpointing"] = True

args = TrainingArguments(**ta_kwargs)

# ============================================================
# DPOTrainer
# ============================================================
# beta controls strength of preference; common values 0.05–0.2
BETA = 0.1

trainer = DPOTrainer(
    model=policy,
    ref_model=ref_model,
    args=args,
    beta=BETA,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tok,
    max_length=1024,
    max_prompt_length=768,
)

trainer.train()

# ============================================================
# Save DPO adapter + tokenizer
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)
trainer.model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print(f"\nSaved AGENT DPO LoRA adapter + tokenizer to: {OUT_DIR}")

# ============================================================
# Quick held-out evaluation: greedy relaxation accuracy
# (We evaluate at ITEM level, not pair level.)
# ============================================================
@torch.inference_mode()
def agent_predict_relaxation(prompt_text: str, max_new_tokens: int = 96) -> Optional[dict]:
    inputs = tok(prompt_text, return_tensors="pt").to(trainer.model.device)
    out = trainer.model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    gen = tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    obj = extract_json_robust(gen)
    return obj

def normalize_relaxed(obj: Optional[dict]) -> Optional[Tuple[str,str,str]]:
    if not isinstance(obj, dict): 
        return None
    rc = obj.get("relaxed_constraint", None)
    if not isinstance(rc, dict):
        return None
    col = rc.get("column", None)
    op  = rc.get("op", None)
    val = rc.get("value", None)
    if col is None or op is None:
        return None
    return (str(col), str(op), str(val))

def oracle_relaxed_key(it: Dict[str,Any]) -> Optional[Tuple[str,str,str]]:
    o = it.get("chosen_relaxation") or it.get("relaxed_constraint") or it.get("unique_repair_constraint")
    if not isinstance(o, dict):
        return None
    return (str(o.get("column")), str(o.get("op")), str(o.get("value")))

# Build prompts at item-level and score
correct = 0
total = 0
bad_parse = 0

# Evaluate on a subset if you want faster; set to None for full test split
EVAL_LIMIT = None

for idx, it in enumerate(bench_test):
    if EVAL_LIMIT is not None and total >= EVAL_LIMIT:
        break

    base_query = (it.get("base_query_sentence") or "").strip()
    persona = (it.get("persona") or "").strip()
    constraints = it.get("additional_constraints") or []
    okey = oracle_relaxed_key(it)

    if not base_query or not persona or not constraints or okey is None:
        continue

    # only evaluate infeasible full sets
    try:
        base = parse_base_from_sentence(df, base_query)
        full = apply_constraints(df, base, constraints)
        if len(full) > 0:
            continue
    except Exception:
        continue

    prompt_text = build_prompt_text(base_query, persona, constraints)
    pred = agent_predict_relaxation(prompt_text)
    pkey = normalize_relaxed(pred)
    if pkey is None:
        bad_parse += 1

    total += 1
    if pkey == okey:
        correct += 1

acc = (correct / total) if total else 0.0
print("\n=== HELD-OUT RELAXATION ACCURACY (item-level) ===")
print(f"Evaluated items: {total}")
print(f"Correct relaxations: {correct}")
print(f"Accuracy: {acc:.2%}")
print(f"Bad/Unparsed JSON outputs: {bad_parse} ({(bad_parse/total):.2%} if total else 0)")


In [ ]:
# ============================================================
# DPO (QLoRA) for Qwen/Qwen3-8B (API-ROBUST TRL)
# ============================================================

import os, json, inspect, re, random
from pathlib import Path

# ---- GPU (edit if you want) ----
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "5")

# ---- Minimal installs (only if missing) ----
def _pip_install(pkgs):
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

try:
    import torch
    import transformers
except Exception:
    _pip_install(["torch", "transformers"])

try:
    import datasets
except Exception:
    _pip_install(["datasets"])

try:
    import peft
except Exception:
    _pip_install(["peft"])

try:
    import bitsandbytes
except Exception:
    _pip_install(["bitsandbytes"])

try:
    import trl
except Exception:
    _pip_install(["trl"])

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training

# TRL imports (robust)
from trl import DPOTrainer
DPOConfig = getattr(trl, "DPOConfig", None)
create_reference_model = getattr(trl, "create_reference_model", None)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", getattr(trl, "__version__", "unknown"))

# ============================================================
# Config
# ============================================================
MODEL_NAME = "Qwen/Qwen3-8B"

TRAIN_JSONL = "./dpo_pairs_train.jsonl"
TEST_JSONL  = "./dpo_pairs_test.jsonl"

SFT_ADAPTER_DIR = "./qwen3_roleplayer_sft_lora"   # optional init
OUT_DIR = "./qwen3_roleplayer_dpo_lora"

# DPO hyperparams (3090-friendly)
SEED = 11
random.seed(SEED)
torch.manual_seed(SEED)

BETA = 0.1
LR = 5e-5
EPOCHS = 1
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
MAX_LENGTH = 1024
MAX_PROMPT_LENGTH = 768

# ============================================================
# Load JSONL preference pairs (prompt/chosen/rejected)
# ============================================================
for p in [TRAIN_JSONL, TEST_JSONL]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing required file: {p}")

# load_dataset("json") reads JSONL when file ends with .jsonl
train_ds = load_dataset("json", data_files=TRAIN_JSONL, split="train")
test_ds  = load_dataset("json", data_files=TEST_JSONL, split="train")

need_cols = {"prompt", "chosen", "rejected"}
for name, ds in [("train", train_ds), ("test", test_ds)]:
    cols = set(ds.column_names)
    if not need_cols.issubset(cols):
        raise ValueError(f"{name} dataset missing columns. Need {need_cols}, got {cols}")

print(f"Loaded DPO pairs: train={len(train_ds)} test={len(test_ds)}")
print("Example keys:", train_ds.column_names)
print("Example prompt head:", (train_ds[0]["prompt"][:140] + "...") if len(train_ds) else "EMPTY")

# ============================================================
# Tokenizer + policy model (4-bit QLoRA)
# ============================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token

def _load_base_4bit():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )
    m.config.use_cache = False
    return m

policy = _load_base_4bit()
policy = prepare_model_for_kbit_training(policy, use_gradient_checkpointing=True)

# If SFT adapter exists, load it first (continue training the same adapter weights)
if Path(SFT_ADAPTER_DIR).exists():
    print(f"Loading SFT adapter into policy: {SFT_ADAPTER_DIR}")
    policy = PeftModel.from_pretrained(policy, SFT_ADAPTER_DIR, is_trainable=True)
else:
    print(f"No SFT adapter found at {SFT_ADAPTER_DIR} (continuing from base).")

# If model isn't already a PEFT model (no SFT adapter), add a LoRA adapter for DPO
if not hasattr(policy, "peft_config"):
    def guess_lora_targets(model):
        preferred = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        found = set()
        for name, module in model.named_modules():
            if module.__class__.__name__ == "Linear":
                last = name.split(".")[-1]
                if last in preferred:
                    found.add(last)
        return sorted(found) if found else preferred

    target_modules = guess_lora_targets(policy)
    print("LoRA target_modules:", target_modules)
    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    policy = get_peft_model(policy, lora_cfg)

# Print trainable params
try:
    policy.print_trainable_parameters()
except Exception:
    # fallback
    trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    total = sum(p.numel() for p in policy.parameters())
    print(f"trainable params: {trainable:,} || all params: {total:,} || trainable%: {100*trainable/total:.4f}")

# ============================================================
# Reference model (frozen)
# ============================================================
# Best: TRL helper to clone & freeze (if available)
if callable(create_reference_model):
    ref_model = create_reference_model(policy)
    print("Created ref_model via trl.create_reference_model(policy)")
else:
    # Fallback: load another copy + optional SFT adapter, freeze it
    print("create_reference_model not found; loading a second frozen ref_model.")
    ref_model = _load_base_4bit()
    if Path(SFT_ADAPTER_DIR).exists():
        ref_model = PeftModel.from_pretrained(ref_model, SFT_ADAPTER_DIR, is_trainable=False)
    for p in ref_model.parameters():
        p.requires_grad = False

# ============================================================
# TrainingArguments / DPOConfig (version-robust)
# ============================================================
def _version_tuple(v: str):
    nums = [int(x) for x in re.findall(r"\d+", (v or "0").split("+")[0])]
    while len(nums) < 3: nums.append(0)
    return tuple(nums[:3])

# transformers changed evaluation_strategy -> eval_strategy in some versions
ta_sig = inspect.signature(TrainingArguments.__init__)
has_eval_strategy = "eval_strategy" in ta_sig.parameters
has_evaluation_strategy = "evaluation_strategy" in ta_sig.parameters
has_report_to = "report_to" in ta_sig.parameters
has_optim = "optim" in ta_sig.parameters

ta_kwargs = dict(
    output_dir=OUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    fp16=True,
)

if has_eval_strategy:
    ta_kwargs["eval_strategy"] = "steps"
elif has_evaluation_strategy:
    ta_kwargs["evaluation_strategy"] = "steps"

ta_kwargs["save_strategy"] = "steps"
if has_report_to:
    ta_kwargs["report_to"] = "none"
if has_optim:
    # guard: some envs don't have paged optimizer string; TRL/BNB usually does, but be safe
    ta_kwargs["optim"] = "paged_adamw_32bit"

# If TRL provides DPOConfig, use it (newer API: beta/max_length often live here)
if DPOConfig is not None:
    dpo_sig = inspect.signature(DPOConfig.__init__)
    dpo_kwargs = {}
    # pass TrainingArguments-like fields that DPOConfig supports
    for k, v in ta_kwargs.items():
        if k in dpo_sig.parameters:
            dpo_kwargs[k] = v

    # Add DPO-specific fields if supported
    for k, v in [
        ("beta", BETA),
        ("max_length", MAX_LENGTH),
        ("max_prompt_length", MAX_PROMPT_LENGTH),
    ]:
        if k in dpo_sig.parameters:
            dpo_kwargs[k] = v

    # some versions use "max_completion_length" instead of deriving from max_length - max_prompt_length
    if "max_completion_length" in dpo_sig.parameters and "max_length" in dpo_sig.parameters and "max_prompt_length" in dpo_sig.parameters:
        # leave unset; trainer can derive; set only if you want explicit:
        pass

    args = DPOConfig(**dpo_kwargs)
    print("Using TRL DPOConfig with fields:", sorted(dpo_kwargs.keys()))
else:
    args = TrainingArguments(**ta_kwargs)
    print("Using transformers.TrainingArguments (no DPOConfig in this TRL).")

# ============================================================
# Build DPOTrainer kwargs (signature-robust)
# ============================================================
trainer_sig = inspect.signature(DPOTrainer.__init__)
trainer_kwargs = {}

# Required-ish
if "model" in trainer_sig.parameters:
    trainer_kwargs["model"] = policy
if "ref_model" in trainer_sig.parameters:
    trainer_kwargs["ref_model"] = ref_model
if "args" in trainer_sig.parameters:
    trainer_kwargs["args"] = args
if "train_dataset" in trainer_sig.parameters:
    trainer_kwargs["train_dataset"] = train_ds
if "eval_dataset" in trainer_sig.parameters:
    trainer_kwargs["eval_dataset"] = test_ds

# Tokenizer arg name changed in some TRL versions
if "processing_class" in trainer_sig.parameters:
    trainer_kwargs["processing_class"] = tok
elif "tokenizer" in trainer_sig.parameters:
    trainer_kwargs["tokenizer"] = tok

# Older TRL put these in __init__; newer TRL expects them in DPOConfig/args
if "beta" in trainer_sig.parameters:
    trainer_kwargs["beta"] = BETA
if "max_length" in trainer_sig.parameters:
    trainer_kwargs["max_length"] = MAX_LENGTH
if "max_prompt_length" in trainer_sig.parameters:
    trainer_kwargs["max_prompt_length"] = MAX_PROMPT_LENGTH

# Some TRL versions accept "max_target_length" or similar — leave alone unless present
# (we won’t guess extra params to avoid more TypeErrors)

print("DPOTrainer will be constructed with kwargs:", sorted(trainer_kwargs.keys()))

trainer = DPOTrainer(**trainer_kwargs)

# ============================================================
# Train
# ============================================================
trainer.train()

# ============================================================
# Save adapter + tokenizer
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)

# If policy is PEFT model, saving stores the adapter; otherwise saves full model (not expected here)
try:
    trainer.model.save_pretrained(OUT_DIR)
except Exception:
    # sometimes trainer wraps; fallback
    policy.save_pretrained(OUT_DIR)

tok.save_pretrained(OUT_DIR)
print(f"\nSaved DPO adapter + tokenizer to: {OUT_DIR}")

# ============================================================
# Quick sanity: score chosen vs rejected on a few examples (optional vibe check)
# ============================================================
@torch.no_grad()
def _logprob(model, text: str) -> float:
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model(**enc)
    logits = out.logits[:, :-1, :]
    labels = enc["input_ids"][:, 1:]
    logp = torch.log_softmax(logits, dim=-1).gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    return float(logp.sum().item())

def _maybe_concat(prompt, completion):
    # If your JSONL already contains fully templated strings, you may want concat.
    # Many DPO datasets store prompt separately and chosen/rejected as assistant-only.
    # Here we assume you stored strings ready to be concatenated.
    return prompt + completion

policy.eval()
ncheck = min(5, len(test_ds))
print("\n--- VIBE CHECK (policy prefers chosen?) ---")
for i in range(ncheck):
    ex = test_ds[i]
    # Try both ways: if chosen already includes prompt, then concatenation double-counts.
    # We'll detect by simple substring:
    if ex["prompt"].strip() and ex["prompt"].strip() in ex["chosen"]:
        chosen_text = ex["chosen"]
        rejected_text = ex["rejected"]
    else:
        chosen_text = _maybe_concat(ex["prompt"], ex["chosen"])
        rejected_text = _maybe_concat(ex["prompt"], ex["rejected"])

    lp_c = _logprob(policy, chosen_text)
    lp_r = _logprob(policy, rejected_text)
    print(f"ex {i}: logp(chosen)={lp_c:.1f} | logp(rejected)={lp_r:.1f} | prefers_chosen={lp_c>=lp_r}")
